In [ ]:
library(Seurat)

library(reticulate)
library(anndata)
library(ggplot2)
library(ggrepel)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(repr)
library(scico)
options(repr.plot.width=10, repr.plot.height=8)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("all", "old", "young")
colorSamples <- c("old"="#A58065",
                "young"="#7FCFF2", 
                "all"="#92A8AC")
dir.create(paste0("figures_", dataset_id))
for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c(
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

thrMinCells <- 500 * 0.05

In [ ]:
theme_paper <- function(base_size = 17, base_family = "") {
  theme_minimal(base_size = base_size, base_family = base_family) +
    theme(
      plot.title = element_text(face = "bold", size = base_size + 2, hjust = 0.5),
      axis.title = element_text(size = base_size),
      axis.text  = element_text(size = base_size * 0.9),
      legend.title = element_text(size = base_size),
      legend.text  = element_text(size = base_size * 0.9),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_blank(),
      plot.margin = margin(10, 10, 10, 10)
    )
}

In [ ]:
load("workspaces/00_evaluation_objectCreation.Rdata")

In [ ]:
# compute TE length
conversion_table$TElength <- conversion_table$end - conversion_table$start

In [ ]:
# load truncation annotation (uploaded to Zenodo)
annotation_truncation <- read.table("annotation/annotated_tes_mm10.tsv", header = TRUE)

table(annotation_truncation$status)
table(annotation_truncation$annotation)

# Full_Length — the insertion covers ≥90% of the repeat consensus sequence
# 5prime_Truncated — >10% of the consensus is missing from the 5' end only
# 3prime_Truncated — >10% of the consensus is missing from the 3' end only
# Internal_Fragment — >10% missing from both ends; only the middle of the element is present
# Truncated_Other — coverage <90% but neither end gap exceeds 10%; typically a slightly degraded or ambiguously trimmed element

colnames(annotation_truncation) <- gsub(colnames(annotation_truncation), pattern="genoName", replacement="chr")
colnames(annotation_truncation) <- gsub(colnames(annotation_truncation), pattern="genoStart", replacement="start")
colnames(annotation_truncation) <- gsub(colnames(annotation_truncation), pattern="genoEnd", replacement="end")
annotation_truncation$start <- annotation_truncation$start + 1
colnames(annotation_truncation)

In [ ]:
# add truncation to the TE annotation table
conversion_table <- merge(conversion_table, 
                        annotation_truncation[,c("chr","start","end","strand","coverage","status", "annotation")], 
                        all.x=T)
dim(conversion_table)

In [ ]:
annotationYoungSim <- conversion_table[conversion_table$stellarscopeID %in% rownames(splatter_objs[["young"]]),]
annotationOldSim <- conversion_table[conversion_table$stellarscopeID %in% rownames(splatter_objs[["old"]]),]

# Locus characteristics

In [ ]:
familyOrder <- c("L1",'ERVL','ERVK','ERV1','ERVL-MaLR','B2','Alu')

annotationYoungSim$family <- factor(annotationYoungSim$family, levels = rev(familyOrder))
  
# simulated loci

options(repr.plot.width=8, repr.plot.height=4)
ggplot(annotationYoungSim, aes(x=propSubs, y=family, fill=family)) +
    geom_boxplot(alpha=0.5) +
    scale_fill_scico_d(palette="lipari") +
    guides(fill = "none") +
    theme_pubr() +
    theme(text=element_text(size=15)) +
    ggtitle("Proportion of substitutions in simulated young loci by family")
table(annotationYoungSim$family)
ggsave("figures_simulated_mm_RA_young/distributionOfProbSub.pdf", device="pdf", width=8, height=4)

options(repr.plot.width=8, repr.plot.height=4)
ggplot(annotationYoungSim, aes(x=TElength, y=family, fill=family)) +
    geom_boxplot(alpha=0.5) +
    scale_fill_scico_d(palette="lipari") +
    guides(fill = "none") +
    theme_pubr() +
    theme(text=element_text(size=15)) +
    ggtitle("Length of simulated young loci by family")
table(annotationYoungSim$family)
ggsave("figures_simulated_mm_RA_young/distributionOfTElengths.pdf", device="pdf", width=8, height=6)


In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
ggplot(as.data.frame(table(annotationYoungSim$status, annotationYoungSim$family)), aes(x=Freq, y=Var2, fill=Var1)) +
    ylab("family") +
    xlab("N. simulated TEs") +
    labs(fill = "Truncation status") +
    scale_fill_manual(values=c("grey5", "darkcyan")) +
    
    geom_col(alpha = 0.8) +
    theme_paper()
ggsave("figures_simulated_mm_RA_young/distributionOfTruncationStatus_allYoung.pdf", device="pdf", width=8, height=4)

options(repr.plot.width=10, repr.plot.height=4)
ggplot(as.data.frame(table(annotationYoungSim$annotation, annotationYoungSim$family)), aes(x=Freq, y=Var2, fill=Var1)) +
    ylab("family") +
    labs(fill = "Truncation type") +
    xlab("N. simulated TEs") +
    geom_col(alpha = 0.8) +
    scale_fill_scico_d(palette="berlin") +
    theme_paper()
ggsave("figures_simulated_mm_RA_young/distributionOfTruncationType_allYoung.pdf", device="pdf", width=10, height=4)

# Evaluation

## Upset plots of detected TEs and TPs

### Upsets of detected TEs

In [ ]:
options(repr.plot.width=12, repr.plot.height=6)

TPlistSample <- list()
FPlistSample <- list()
FNlistSample <- list()

for (sample in samples){

    print(sample)
    
    detectedList <- list()

    TPlistSample[[sample]] <- list()
    FPlistSample[[sample]] <- list()
    FNlistSample[[sample]] <- list()

    for (tool in c(names(objList), "simulated")){

        print(tool)
        if (tool == "simulated") {
            obj <- splatter_objs[[sample]]
        }else {
            obj <- objList[[tool]][[sample]]
        }

        detectedList[[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- Features(obj)
        TPlistSample[[sample]][[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- intersect(Features(splatter_objs[[sample]]), Features(obj))
        FPlistSample[[sample]][[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- setdiff(Features(obj), Features(splatter_objs[[sample]]))
        FNlistSample[[sample]][[sub("^(.*)_.*$", "\\1", obj@project.name)]] <- setdiff(Features(splatter_objs[[sample]]), Features(obj))
    }

    #pdf(file=paste0("figures_",dataset_id,"_",sample,"/upset_detected.pdf"), width = 13, height = 7.5)
    show(UpSetR::upset(fromList(detectedList), nintersects = 30,
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.5, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(detectedList))+1000, #show.numbers = F,
            nsets=length(objList)+1,
            sets.x.label="N. detected loci",
            mainbar.y.label="Intersection size")
        
    )
    grid.text(paste0("Detected TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    #dev.off()

    #pdf(file=paste0("figures_",dataset_id,"_",sample,"/upset_detected_TP.pdf"), width = 11, height = 7.5)
    show(UpSetR::upset(fromList(TPlistSample[[sample]]), nintersects = 20, 
            sets=names(colorTools), keep.order = T, sets.bar.color=colorTools, 
            text.scale = c(2, 2, 2, 1.65, 2.2, 1.5), 
            order.by=c("freq"),
            point.size=3, mb.ratio = c(0.5, 0.5),
            set_size.show=F, set_size.scale_max= max(lengths(TPlistSample[[sample]]))+1000, #show.numbers = F, 
            nsets=length(c(objList, splatter_obj)),
            sets.x.label="N. correctly detected loci",
            mainbar.y.label="Intersection size")
    )
    grid.text(paste0("TP TEs - ", sample ),x = 0.65, y=0.95, gp=gpar(fontsize=18))
    
    #dev.off()
}


### Common biases

In [ ]:
sample <- "young"

# get common false positives
FPlistSample[[sample]]["simulated"] <- NULL
lengths(FPlistSample[[sample]])
commonFPs <- Reduce(intersect, FPlistSample[[sample]])
length(commonFPs)

# get common false negatives
FNlistSample[[sample]]["simulated"] <- NULL
lengths(FNlistSample[[sample]])
commonFNs <- Reduce(intersect, FNlistSample[[sample]])
length(commonFNs)


In [ ]:
fp <- conversion_table %>% filter(stellarscopeID %in% commonFPs)
tp <- conversion_table %>% filter(stellarscopeID %in% rownames(splatter_objs[["young"]]))
fn <- conversion_table %>% filter(stellarscopeID %in% commonFNs)

fp$label <- "common FPs"
tp$label <- "simulated loci"
fn$label <- "common FNs"

In [ ]:
options(repr.plot.width=5, repr.plot.height=6)

library(scico)
library(ggbeeswarm)

df <- rbind(fp, tp, fn)
df$label <- factor(df$label, levels=c("simulated loci", "common FNs", "common FPs"))

ggplot(df, aes(label, TElength)) + 
  geom_boxplot() +
  stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
             geom = "text", vjust = -0.5, size = 5) +
  theme_light() +
  xlab("") +
  theme(text=element_text(size=18), 
  axis.text.x=element_text(angle=30, hjust=1))
ggsave(paste0("figures_",dataset_id,"_",sample,"/TElength_commonFP_commonFN.pdf"), width = 4, height = 4)

ggplot(df, aes(x=label, y=TElength)) + 
  geom_violin() +
  stat_summary(
    fun.data = function(x) data.frame(y = max(x), label = sprintf("n: %d", length(x))),
    geom = "text", 
    vjust = -0.5, 
    size = 5
  ) +
  stat_summary(
    fun = median, 
    geom = "crossbar", 
    width = 0.2, 
    color = "grey30", 
    linewidth = 0.5
  ) + 
  stat_summary(
    fun = median,
    geom = "text",
    aes(label = round(after_stat(y), 2)),
    hjust = -0.4,   # Positive values shift text right; negative values shift it left
    vjust = 0.5,    # Centers the text vertically with the crossbar line
    size = 5,
    color = "grey30"
  ) +
  theme_pubr() +
  xlab("") +
  theme(
    text = element_text(size = 18), 
    axis.text.x = element_text(angle = 30, hjust = 1)
  )
ggsave(paste0("figures_",dataset_id,"_",sample,"/TElength_commonFP_commonFN_violin.pdf"), width = 5, height = 6)

ggplot(df, aes(x=label, y=propSubs)) + 
  geom_boxplot() +
  stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
             geom = "text", vjust = -0.5, size = 5) +
  theme_light() +
  xlab("") +
  theme(text=element_text(size=18), 
  axis.text.x=element_text(angle=30, hjust=1))
ggsave(paste0("figures_",dataset_id,"_",sample,"/propSubs_commonFP_commonFN.pdf"), width = 4, height = 4)

ggplot(df, aes(label, propSubs)) + 
  geom_violin() +
    stat_summary(
    fun.data = function(x) data.frame(y = max(x), label = sprintf("n: %d", length(x))),
    geom = "text", 
    vjust = -0.5, 
    size = 5
  ) +
  stat_summary(
    fun = median, 
    geom = "crossbar", 
    width = 0.2, 
    color = "grey30", 
    linewidth = 0.5
  ) + 
  stat_summary(
    fun = median,
    geom = "text",
    aes(label = round(after_stat(y), 2)),
    hjust = -0.4,   # Positive values shift text right; negative values shift it left
    vjust = 0.5,    # Centers the text vertically with the crossbar line
    size = 5,
    color = "grey30"
  ) +
  theme_pubr() +
  xlab("") +
  theme(
    text = element_text(size = 18), 
    axis.text.x = element_text(angle = 30, hjust = 1)
  )
ggsave(paste0("figures_",dataset_id,"_",sample,"/propSubs_commonFP_commonFN_violin.pdf"), width = 4, height = 4)

ggplot(df, aes(label, mya)) + 
  geom_boxplot() +
  stat_summary(fun.data = function(x) data.frame(y = max(x), label = length(x)),
             geom = "text", vjust = -0.5, size = 5) +
  theme_light() +
  theme(text=element_text(size=18), 
  axis.text.x=element_text(angle=30, hjust=1))
ggsave(paste0("figures_",dataset_id,"_",sample,"/mya_commonFP_commonFN.pdf"), width = 4, height = 4)

ggplot(df, aes(family, fill = label)) +
  geom_bar(position = "fill") +
  theme_pubclean() +
  theme(text=element_text(size=18), 
  axis.text.x = element_text(angle=30, hjust=1))
ggsave(paste0("figures_",dataset_id,"_",sample,"/familyComp_commonFP_commonFN.pdf"), width = 4, height = 4)

options(repr.plot.width=6, repr.plot.height=7)
  ggplot(df, aes(label, fill = family)) +
  geom_bar(position = "fill") +
  scale_fill_scico_d(palette="lipari") +
  xlab("") +
  theme_pubr() +
  theme(text=element_text(size=18), 
  axis.text.x = element_text(angle=30, hjust=1))
ggsave(paste0("figures_",dataset_id,"_",sample,"/familyEnrichment_commonFP_commonFN.pdf"), width = 6, height = 6)

#### Common FPs

In [ ]:
# common FPs
df <- rbind(fp, tp)

df %>%
  group_by(label) %>%
  summarise(
    n = n(),
    median_length = median(TElength),
    median_propSubs = median(propSubs),
    mean_age = mean(mya)
  )

wilcox.test(TElength ~ label, data = df)
wilcox.test(propSubs ~ label, data = df)
wilcox.test(mya ~ label, data = df)

table(df$family, df$label)
chisq.test(table(df$family, df$label))

In [ ]:
library(rstatix)
wilcox_effsize(df, TElength ~ label)
wilcox_effsize(df, propSubs ~ label)
wilcox_effsize(df, mya ~ label)

### Common FNs

In [ ]:
# common FNs
df <- rbind(fn, tp)

df %>%
  group_by(label) %>%
  summarise(
    n = n(),
    median_length = median(TElength),
    median_propSubs = median(propSubs),
    mean_age = mean(mya)
  )

wilcox.test(TElength ~ label, data = df)
wilcox.test(propSubs ~ label, data = df)
wilcox.test(mya ~ label, data = df)

table(df$family, df$label)
chisq.test(table(df$family, df$label))

In [ ]:
library(rstatix)
wilcox_effsize(df, TElength ~ label)
wilcox_effsize(df, propSubs ~ label)
wilcox_effsize(df, mya ~ label)


#### Common FP and FN chi square test

In [ ]:
df <- rbind(fp, tp, fn)

tab <- table(df$family, df$label)
tab

chisq <- chisq.test(tab)

chisq

chisq$expected

chisq$stdres

library(rstatix)
cramer_v(tab)


## Precision / Recall


### Thr 0 counts

In [ ]:
precisionTool <- list()
recallTool <- list()
# TPtool <-list()
# FPtool <-list()
# FNtool <- list()

for (sample in samples){
    layer <- "counts"

    matSplatter <- GetAssayData(splatter_objs[[sample]], layer=layer)

    precisionTool[[sample]] <- list()
    recallTool[[sample]] <- list()
    # FPtool[[sample]] <- list()
    # FNtool[[sample]] <- list()

    for(tool in names(objList)){
        
        matTool <- GetAssayData(objList[[tool]][[sample]], layer=layer)
    
        precisionTool[[sample]][[tool]] <- NULL
        recallTool[[sample]][[tool]] <- NULL
        
        for(cell in Cells(splatter_objs[[sample]])){
            
            exprSplatter <- names(matSplatter[,cell])[matSplatter[,cell] > 0] 
            # TEs expressed in the simulated matrix
            
            exprTool <- names(matTool[,cell])[matTool[,cell] > 0] 
            # TEs detected by the tool

            TP <- intersect(exprSplatter, exprTool)
            #TPtool[[sample]][[tool]] <- TP
            nTP <- length(TP)
            
            FP <- setdiff(exprTool, exprSplatter)
            #FPtool[[sample]][[tool]] <- FP
            nFP <- length(FP)
            
            # add FP genes to the FP count
            # if(tool %in% names(objGeneList)){
                   #     FP_genes <- names(matTool_genes[,cell])[matTool_genes[,cell] > 0]
            #     nFP_genes <- length(FP_genes)
            #     nFP <- nFP + nFP_genes
            # }
            
            FN <- setdiff(exprSplatter, exprTool)
            # FNtool[[sample]][[tool]] <- FN
            nFN <- length(FN)
            
            precision <- nTP / (nTP + nFP)
            precisionTool[[sample]][[tool]] <- c(precisionTool[[sample]][[tool]], precision)
            
            recall <- nTP / (nTP + nFN)
            recallTool[[sample]][[tool]] <- c(recallTool[[sample]][[tool]], recall)    
        }
    }
}

In [ ]:
options(repr.plot.width=5, repr.plot.height=3)

for (sample in samples){
    print(sample)

    precisionDf <- stack(precisionTool[[sample]])
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(recallTool[[sample]])
    colnames(recallDf) <- c("recall", "tool")

    df <- merge(precisionDf, recallDf, by='tool')

    df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))

    # plot just the mean
    precisionDf <- stack(lapply(precisionTool[[sample]], mean))
    colnames(precisionDf) <- c("precision", "tool")

    recallDf <- stack(lapply(recallTool[[sample]], mean))
    colnames(recallDf) <- c("recall", "tool")

    Fscores <- sapply(names(precisionTool[[sample]]), function(tool){
                            (2 * precisionTool[[sample]][[tool]] * recallTool[[sample]][[tool]]) / 
                            (precisionTool[[sample]][[tool]] + recallTool[[sample]][[tool]])
    })
    meanFscores <- colMeans(Fscores)
    FscoreDf <- stack(meanFscores)
    colnames(FscoreDf) <- c("F1score", "tool")
    
    df <- data.frame(
        tool = rep(names(precisionTool[[sample]]),
                    times = sapply(precisionTool[[sample]], length)),
        precision = unlist(precisionTool[[sample]]),
        recall    = unlist(recallTool[[sample]])
    )
    df$F1score <- (2 * df$precision * df$recall) / (df$precision + df$recall)

    df$tool <- factor(df$tool, levels=rev(names(colorTools)))

}


In [ ]:
# save just means, quartiles and min max

meanF1score_age_df <- NULL

for(age in c("old", "young")){
  
  precisionDf <- stack(lapply(precisionTool[[age]], mean))
  colnames(precisionDf) <- c("precision", "tool") # precision is mean precision
  precisionDf$precision_min <- stack(lapply(precisionTool[[age]], min))$values
  precisionDf$precision_max <- stack(lapply(precisionTool[[age]], max))$values
  precisionDf$precision_median <- stack(lapply(precisionTool[[age]], median))$values
  precisionDf$precision_sd <- stack(lapply(precisionTool[[age]], sd))$values
  precisionDf$precision_q25 <- stack(lapply(precisionTool[[age]], function(x) quantile(x, probs = 0.25)))$values
  precisionDf$precision_q75 <- stack(lapply(precisionTool[[age]], function(x) quantile(x, probs = 0.75)))$values

  recallDf <- stack(lapply(recallTool[[age]], mean))
  colnames(recallDf) <- c("recall", "tool")
  recallDf$recall_min <- stack(lapply(recallTool[[age]], min))$values
  recallDf$recall_max <- stack(lapply(recallTool[[age]], max))$values
  recallDf$recall_median <- stack(lapply(recallTool[[age]], median))$values
  recallDf$recall_sd <- stack(lapply(recallTool[[age]], sd))$values
  recallDf$recall_q25 <- stack(lapply(recallTool[[age]], function(x) quantile(x, probs = 0.25)))$values
  recallDf$recall_q75 <- stack(lapply(recallTool[[age]], function(x) quantile(x, probs = 0.75)))$values


  Fscores <- sapply(names(precisionTool[[age]]), function(tool){
                          (2 * precisionTool[[age]][[tool]] * recallTool[[age]][[tool]]) /
                          (precisionTool[[age]][[tool]] + recallTool[[age]][[tool]])
                          
  })
  meanFscores <- colMeans(Fscores)
  FscoreDf <- stack(meanFscores)
  colnames(FscoreDf) <- c("F1score", "tool")
  FscoreDf$F1score_min <- apply(Fscores, MARGIN = 2, min)
  FscoreDf$F1score_max <- apply(Fscores, MARGIN = 2, max)
  FscoreDf$F1score_median <- apply(Fscores, MARGIN = 2, median)
  FscoreDf$F1score_sd <- apply(Fscores, MARGIN = 2, sd)
  FscoreDf$F1score_q25 <- apply(Fscores, MARGIN = 2,function(x) quantile(x, probs = 0.25))
  FscoreDf$F1score_q75 <- apply(Fscores, MARGIN = 2,function(x) quantile(x, probs = 0.75))

  df <- merge(precisionDf, recallDf, by=c("tool"))
  df <- merge(df, FscoreDf, by=c("tool"))

  df$tool <- factor(df$tool, levels=rev(names(colorTools)))
  
  # create df for facet plot
  meanF1score_age_df <- rbind(meanF1score_age_df, cbind(df, age)) 

}


In [ ]:
meanF1score_age_df_forDepthcomparison <- meanF1score_age_df

meanF1score_age_df_forDepthcomparison <- meanF1score_age_df_forDepthcomparison[!(
    meanF1score_age_df_forDepthcomparison$tool == "Stellarscope" & 
    meanF1score_age_df_forDepthcomparison$age == "young"),] 

meanF1score_age_df_forDepthcomparison$tool[meanF1score_age_df_forDepthcomparison$tool == "Stellarscope_manualrun"] <- "Stellarscope"
meanF1score_age_df_forDepthcomparison

write.table(meanF1score_age_df_forDepthcomparison, "data/meanF1score_age_df.tsv")

In [ ]:
options(repr.plot.width=9, repr.plot.height=4)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 2) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
ggsave(paste0("figures_",dataset_id,"/meanF1_tool_byAge_horiz.png"), height=3.2, width=7)
ggsave(paste0("figures_",dataset_id,"/meanF1_tool_byAge_horiz.pdf"), device="pdf", height=3.2, width=7)

In [ ]:
options(repr.plot.width=5, repr.plot.height=6)

ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1 score")) +
    guides(fill="none", color="none") +
  xlim(c(0,1)) +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))
ggsave(paste0("figures_",dataset_id,"/meanF1_tool_byAge_vert.pdf"), device="pdf", height=6, width=5)

In [ ]:
options(repr.plot.width=8, repr.plot.height=6)

precision_plot <- ggplot(meanF1score_age_df, aes(x=precision, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean precision")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_text(size=16), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

recall_plot <- ggplot(meanF1score_age_df, aes(x=recall, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean recall")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_blank(), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

f1_plot <- ggplot(meanF1score_age_df, aes(x=F1score, y=tool, color=tool, fill=tool)) +
    facet_wrap(~age, ncol = 1) +
    geom_col(alpha = 0.9) +
    scale_color_manual(values=colorTools) +
    scale_fill_manual(values=colorTools) +
    theme_pubclean() + ggtitle(paste0("Mean F1score")) +
    guides(fill="none", color="none") +
    xlim(c(0,1)) +
    ylab("") +
    theme(text=element_text(size=17), 
      plot.title = element_text(size=18, hjust=0.5), 
      plot.subtitle = element_text(size=18, hjust=0.5), 
      axis.text.y = element_blank(), 
      axis.text.x = element_text(size=13), 
      strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
      strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
      panel.spacing = unit(1, "lines"))

plot <- precision_plot + recall_plot
plot

ggsave(paste0("figures_simulated_mm_RA/meanPrecisionRecall_tool_byAge.pdf"), 
      device="pdf", height=6, width=8)

In [ ]:
options(repr.plot.width=11, repr.plot.height=6)

plot + f1_plot
ggsave(paste0("figures_simulated_mm_RA/meanPrecisionRecallF1score_tool_byAge.pdf"), 
      device="pdf", height=6, width=11)

In [ ]:
saveRDS(objList, paste0("data/objList_", dataset_id, ".RDS"))

# Precision recall curve

In [ ]:
pad_sparse_matrix <- function(mat, all_genes) {
  missing_genes <- setdiff(all_genes, rownames(mat))
  if (length(missing_genes) > 0) {
    zero_mat <- Matrix(0, nrow = length(missing_genes), ncol = ncol(mat), sparse = TRUE,
                       dimnames = list(missing_genes, colnames(mat)))
    mat <- rbind(mat, zero_mat)
  }
  # Reorder rows to match all_genes
  mat <- mat[all_genes, , drop = FALSE]
  return(mat)
}

In [ ]:
precisionRecallDfTool <- list()
prCurves <- list()
rocCurves <- list()
auprc_df <- list() 

for(sample in samples){

        auprc_df[[sample]] <- data.frame()

        precisionRecallDfTool[[sample]] <- NULL

        gt_matrix <- GetAssayData(splatter_objs[[sample]], layer='count')

        prCurves[[sample]] <- list()
        rocCurves[[sample]] <- list()

        for(tool in names(objList)){
                
                inferred_matrix <- GetAssayData(objList[[tool]][[sample]], layer='count')
                gc()
                
                # add missing loci to both matrices
                all_genes <- union(rownames(gt_matrix), rownames(inferred_matrix))
                gt_padded <- pad_sparse_matrix(gt_matrix, all_genes)
                inferred_padded <- pad_sparse_matrix(inferred_matrix, all_genes)

                # Flatten to vectors
                gt_vec <- as.vector(gt_padded)
                inferred_vec <- as.vector(inferred_padded)

                # Binarize ground truth: 1 if expressed, 0 otherwise
                gt_binary <- as.numeric(gt_vec > 0)
        
                # Positive class scores: inferred values for ground truth = 1
                # Negative class scores: inferred values for ground truth = 0
                pos_scores <- inferred_vec[gt_binary == 1]
                neg_scores <- inferred_vec[gt_binary == 0]

                # Create PR curve
                pr_obj <- pr.curve(
                        scores.class0 = inferred_vec,
                        weights.class0 = gt_binary,
                        curve = TRUE,
                        max.compute = TRUE,
                        min.compute = TRUE,
                        rand.compute = TRUE
                )
                prCurves[[sample]][[tool]] <- pr_obj

                rocCurves[[sample]][[tool]] <- roc.curve(scores.class0 = inferred_vec, weights.class0 = gt_binary, curve = TRUE,
                        max.compute = TRUE, min.compute = TRUE, rand.compute = TRUE)
                

                auprc <- pr_obj$auc.integral

                auprc_df[[sample]] <- rbind(
                                        auprc_df[[sample]],
                                        data.frame(
                                                sample = sample,
                                                tool = tool,
                                                AUPRC = auprc
                                        )
                                )
                
                # # Manually plot curves
                thresholds <- c(1e-10, seq(1, 9, length.out = 9), seq(10, 100, length.out = 10))
                precision <- numeric(length(thresholds))
                recall <- numeric(length(thresholds))
                F1score <- numeric(length(thresholds))
                # Create binary labels: 1 = expressed, 0 = not expressed
                # Based on ground truth
                labels <- ifelse(gt_binary > 0, 1, 0)

                for (i in seq_along(thresholds)) {
                        t <- thresholds[i]
                        pred_binary <- ifelse(inferred_vec >= t, 1, 0)

                        TP <- sum(pred_binary == 1 & labels == 1)
                        FP <- sum(pred_binary == 1 & labels == 0)
                        FN <- sum(pred_binary == 0 & labels == 1)

                        precision[i] <- ifelse(TP + FP == 0, 1, TP / (TP + FP))
                        recall[i] <- ifelse(TP + FN == 0, 0, TP / (TP + FN))
                        F1score[i] <- ( 2 * precision[i] * recall[i] ) / ( precision[i] + recall[i] )
                }
                
                df_plot <- as.data.frame(cbind(precision, recall, F1score, thresholds))
                mini_df <- df_plot[1:5,]


                df_plot <- cbind(df_plot, tool)
                precisionRecallDfTool[[sample]] <- rbind(precisionRecallDfTool[[sample]], df_plot)
        }
}


In [ ]:
options(repr.plot.width=6, repr.plot.height=4.5)
df <- bind_rows(auprc_df)
df$tool <- factor(df$tool, levels = names(colorTools))
ggplot(df, aes(tool, AUPRC)) +
  geom_boxplot(aes(fill=tool), alpha = 0.3) +
  geom_jitter(width = 0, size=3, alpha = 1, aes(color=sample)) +
  scale_color_manual(values = colorSamples) +
  scale_fill_manual(values = colorTools) +
  ggtitle("AUPRC in each simulation") + 
  guides(fill = "none") +
  theme_minimal() +
  theme(text=element_text(size=18),
          plot.title = element_text(hjust=0.5), 
          plot.subtitle = element_text(hjust=0.5), 
          axis.text.y = element_text(size=16), 
          axis.text.x = element_text(size=16, angle=45, hjust=1),
          panel.grid.major = element_line(colour = "grey90", linewidth = 0.4),
          panel.grid.minor = element_line(colour = "white", linewidth = 0.2),
          legend.position="right")
ggsave(paste0("figures_", dataset_id,"/AUPRC_by_toolAndSim.pdf"), width = 6, height= 4.5)

In [ ]:
for(sample in samples){
    precisionRecallDfTool[[sample]]$thresholds <- as.numeric(gsub(precisionRecallDfTool[[sample]]$thresholds, pattern = 1e-10, replacement=0))


    precisionRecallDfTool[[sample]] <- precisionRecallDfTool[[sample]][(precisionRecallDfTool[[sample]]$thresholds!=0) |
                                (precisionRecallDfTool[[sample]]$tool == "STARsolo_TE_EM"), ]
}

In [ ]:
options(repr.plot.width=9, repr.plot.height=7)

for(sample in samples){

        mini_df <- precisionRecallDfTool[[sample]][(precisionRecallDfTool[[sample]]$thresholds %in% c(0, 1,5,10,100)),]

        # Plot manual PR curve
        show( ggplot(precisionRecallDfTool[[sample]], aes(x=recall, y=precision, color=tool)) + 
                geom_line(size=1, alpha=0.8) +
                geom_point(size=0.7) +
                scale_color_manual(values=colorTools) +
                ggrepel::geom_text_repel(data = mini_df, aes(label = thresholds), size= 5) +
                xlim(c(0,1)) +
                ylim(c(0,1)) +
                theme_pubr() +
                theme(text=element_text(size=20),
                        plot.title = element_text(hjust=0.5), 
                        plot.subtitle = element_text(hjust=0.5), 
                        axis.text.y = element_text(size=15), 
                        axis.text.x = element_text(size=15),
                        panel.grid.major = element_line(colour = "grey90", linewidth = 0.3),
                        panel.grid.minor = element_line(colour = "grey90", linewidth = 0.2),
                        legend.position="right") +
                ggtitle("Precision-Recall Curves", subtitle=paste0(sample," TEs")) )

        ggsave(paste0("figures_simulated_mm_RA_",sample,"/PrecisionRecallCurve_byTool.png"), height=7, width=9)
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/PrecisionRecallCurve_byTool.pdf"), device="pdf", height=7, width=9)
        
}

In [ ]:
options(repr.plot.width=10, repr.plot.height=7)

for(sample in samples){

        # best threshold for each tool
        best_pts <- precisionRecallDfTool[[sample]] %>%
        group_by(tool) %>%
        slice_max(F1score, n = 1, with_ties = FALSE) %>%
        ungroup()

        show(ggplot(precisionRecallDfTool[[sample]], aes(thresholds, F1score, color = tool)) +
                geom_line(size=1, alpha=0.8) +
                #geom_point() +
                scale_color_manual(values=colorTools) +
                #ggrepel::geom_text_repel(data = mini_df, aes(label = thresholds), size= 5) +
                geom_point(data=best_pts, size=3) +
                ggrepel::geom_text_repel(
                        data = best_pts,
                        aes(label = paste0("t=", thresholds)),
                        show.legend = FALSE,
                        size = 5
                        ) +
                ylim(c(0,1)) +
                scale_x_continuous(transform = "sqrt") +
                theme_pubr() +
                theme(text=element_text(size=20),
                        plot.title = element_text(hjust=0.5), 
                        plot.subtitle = element_text(hjust=0.5), 
                        axis.text.y = element_text(size=15), 
                        axis.text.x = element_text(size=15),
                        panel.grid.major = element_line(colour = "grey90", linewidth = 0.3),
                        panel.grid.minor = element_line(colour = "#968787", linewidth = 0.2),
                        legend.position="right") +
                ggtitle("F1 scores", subtitle=paste0(sample," TEs")) )

        ggsave(paste0("figures_simulated_mm_RA_",sample,"/F1scores_byThreshold_byTool.png"), height=7, width=9)
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/F1scores_byThreshold_byTool.pdf"), device="pdf", height=7, width=9)

        show(ggplot(precisionRecallDfTool[[sample]], aes(thresholds, F1score, color = tool)) +
                geom_line(size=1, alpha=0.8) +
                scale_color_manual(values=colorTools) +
                #ggrepel::geom_text_repel(data = mini_df, aes(label = thresholds), size= 5) +
                geom_point(data=best_pts, size=3) +
                ggrepel::geom_text_repel(
                        data = best_pts,
                        aes(label = paste0("t=", thresholds)),
                        show.legend = FALSE,
                        size = 5
                        ) +
                #ylim(c(0,1)) +
                #scale_x_log10() +
                scale_x_continuous(transform = "sqrt") +
                theme_pubr() +
                theme(text=element_text(size=20),
                        plot.title = element_text(hjust=0.5), 
                        plot.subtitle = element_text(hjust=0.5), 
                        axis.text.y = element_text(size=15), 
                        axis.text.x = element_text(size=15),
                        panel.grid.major = element_line(colour = "grey90", linewidth = 0.3),
                        panel.grid.minor = element_line(colour = "#968787", linewidth = 0.2),
                        legend.position="right") +
                ggtitle("F1 scores", subtitle=paste0(sample," TEs")) )
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/F1scores_byThreshold_byTool_noConstraints.pdf"), device="pdf", height=7, width=9)
        
}

In [ ]:
options(repr.plot.width=9, repr.plot.height=7)

for(sample in samples){

        mini_df <- precisionRecallDfTool[[sample]][(precisionRecallDfTool[[sample]]$thresholds %in% c(0,1,5,10,100)),]

        # Plot manual PR curve
        show( ggplot(precisionRecallDfTool[[sample]], aes(x=recall, y=precision, color=tool)) + 
                geom_line(size=1, alpha=0.8) +
                geom_point(size=1) +
                scale_color_manual(values=colorTools) +
                ggrepel::geom_text_repel(data = mini_df, aes(label = thresholds), size=5) +
                # xlim(c(0,1)) +
                # ylim(c(0,1)) +
                theme_pubr() +
                        theme(text=element_text(size=20),
                        plot.title = element_text(hjust=0.5), 
                        plot.subtitle = element_text(hjust=0.5), 
                        axis.text.y = element_text(size=15), 
                        axis.text.x = element_text(size=15),
                        panel.grid.major = element_line(colour = "grey90", linewidth = 0.3),
                        panel.grid.minor = element_line(colour = "#968787", linewidth = 0.2),
                        legend.position="right") +
                ggtitle("Precision-Recall Curve", subtitle=paste0(sample," TEs")) )

        ggsave(paste0("figures_simulated_mm_RA_",sample,"/PrecisionRecallCurve_byTool_noaxis.png"), height=7, width=9)
        ggsave(paste0("figures_simulated_mm_RA_",sample,"/PrecisionRecallCurve_byTool_noaxis.pdf"), device="pdf", height=7, width=9)
        
}